In [5]:
import json
from collections import defaultdict

# baseline_reward_score_csv_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/DiffusionNFT/ckpt-0/evaluation_results.jsonl"
# method_reward_score_csv_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/DiffusionNFT-next-random_9984_images_no_anime_pickscore_002-hpdv3_all-inpainting/ckpt-500/evaluation_results.jsonl"

baseline_reward_score_csv_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/FlowGRPO-PickScore/ckpt-0/evaluation_results.jsonl"
method_reward_score_csv_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/FlowGRPO-PickScore-next-random_9984_images_no_anime_pickscore_002-hpdv3_all-inpainting/ckpt-300/evaluation_results.jsonl"

baseline_data = {}
with open(baseline_reward_score_csv_path, 'r') as f:
    for line in f:
        data = json.loads(line.strip())
        baseline_data[data['sample_id']] = data

method_data = {}
with open(method_reward_score_csv_path, 'r') as f:
    for line in f:
        data = json.loads(line.strip())
        method_data[data['sample_id']] = data

print(f"baseline: {baseline_reward_score_csv_path}")
print(f"method: {method_reward_score_csv_path}")
print(f"Baseline samples: {len(baseline_data)}")
print(f"Method samples: {len(method_data)}")

baseline: /data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/FlowGRPO-PickScore/ckpt-0/evaluation_results.jsonl
method: /data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/FlowGRPO-PickScore-next-random_9984_images_no_anime_pickscore_002-hpdv3_all-inpainting/ckpt-300/evaluation_results.jsonl
Baseline samples: 500
Method samples: 500


In [6]:
if baseline_data and method_data:
    common_sample_ids = set(baseline_data.keys()) & set(method_data.keys())
    print(f"Common sample IDs: {len(common_sample_ids)}")
    
    if common_sample_ids:
        first_id = list(common_sample_ids)[0]
        eval_models = list(baseline_data[first_id]['scores'].keys())
        print(f"Evaluation models: {eval_models}")
    else:
        print("Warning: No common sample IDs found!")
        eval_models = []
else:
    eval_models = []

Common sample IDs: 500
Evaluation models: ['pickscore', 'avg', 'imagereward', 'hpsv3', 'deqa', 'aesthetic_v2_5', 'dinov2', 'q-align', 'aesthetic', 'vila_score', 'unifiedreward', 'ma-agiqa', 'imagedoctor', 'diffdoctor']


In [7]:
win_rates = {}
win_counts = {}
tie_counts = {}
loss_counts = {}
total_counts = {}

# 定义"越小越好"的指标（lower is better）
lower_is_better_models = {'diffdoctor'}

for model_name in eval_models:
    wins = 0
    ties = 0
    losses = 0
    total = 0
    
    # 判断该指标是"越大越好"还是"越小越好"
    lower_is_better = model_name in lower_is_better_models
    
    for sample_id in common_sample_ids:
        if sample_id in baseline_data and sample_id in method_data:
            baseline_score = baseline_data[sample_id]['scores'].get(model_name)
            method_score = method_data[sample_id]['scores'].get(model_name)
            
            if baseline_score is not None and method_score is not None:
                total += 1
                if lower_is_better:
                    # 对于"越小越好"的指标：method_score < baseline_score 是 win
                    if method_score < baseline_score:
                        wins += 1
                    elif method_score == baseline_score:
                        ties += 1
                    else:
                        losses += 1
                else:
                    # 对于"越大越好"的指标：method_score > baseline_score 是 win
                    if method_score > baseline_score:
                        wins += 1
                    elif method_score == baseline_score:
                        ties += 1
                    else:
                        losses += 1
    
    # Win rate 计算：wins 计 1.0，ties 计 0.5，losses 计 0.0
    win_rate = (wins + 0.5 * ties) / total if total > 0 else 0.0
    win_rates[model_name] = win_rate
    win_counts[model_name] = wins
    tie_counts[model_name] = ties
    loss_counts[model_name] = losses
    total_counts[model_name] = total
    
    print(f"{model_name}:")
    print(f"  Win rate: {win_rate:.4f} (({wins} + 0.5 * {ties}) / {total})")
    print(f"  Ties: {ties}, Losses: {losses}")
    print()


pickscore:
  Win rate: 0.5180 ((259 + 0.5 * 0) / 500)
  Ties: 0, Losses: 241

avg:
  Win rate: 0.9880 ((494 + 0.5 * 0) / 500)
  Ties: 0, Losses: 6

imagereward:
  Win rate: 0.5480 ((274 + 0.5 * 0) / 500)
  Ties: 0, Losses: 226

hpsv3:
  Win rate: 0.7480 ((374 + 0.5 * 0) / 500)
  Ties: 0, Losses: 126

deqa:
  Win rate: 0.5150 ((248 + 0.5 * 19) / 500)
  Ties: 19, Losses: 233

aesthetic_v2_5:
  Win rate: 0.4280 ((214 + 0.5 * 0) / 500)
  Ties: 0, Losses: 286

dinov2:
  Win rate: 0.0000 ((0 + 0.5 * 0) / 0)
  Ties: 0, Losses: 0

q-align:
  Win rate: 0.0000 ((0 + 0.5 * 0) / 0)
  Ties: 0, Losses: 0

aesthetic:
  Win rate: 0.6040 ((302 + 0.5 * 0) / 500)
  Ties: 0, Losses: 198

vila_score:
  Win rate: 0.0000 ((0 + 0.5 * 0) / 0)
  Ties: 0, Losses: 0

unifiedreward:
  Win rate: 0.0000 ((0 + 0.5 * 0) / 0)
  Ties: 0, Losses: 0

ma-agiqa:
  Win rate: 0.6020 ((301 + 0.5 * 0) / 500)
  Ties: 0, Losses: 199

imagedoctor:
  Win rate: 0.0000 ((0 + 0.5 * 0) / 0)
  Ties: 0, Losses: 0

diffdoctor:
  Win rate:

In [8]:
# 以表格形式展示结果
import pandas as pd

results_df = pd.DataFrame({
    'Evaluation Model': list(win_rates.keys()),
    'Win Rate': [win_rates[model] for model in win_rates.keys()],
    'Wins': [win_counts[model] for model in win_rates.keys()],
    'Ties': [tie_counts[model] for model in win_rates.keys()],
    'Losses': [loss_counts[model] for model in win_rates.keys()],
    'Total': [total_counts[model] for model in win_rates.keys()],
})

results_df = results_df.sort_values('Win Rate', ascending=False)
print(f"baseline_reward_score_csv_path: {baseline_reward_score_csv_path}")
print(f"method_reward_score_csv_path: {method_reward_score_csv_path}")
print("\nWin Rates Summary:")
print(results_df.to_string(index=False))

baseline_reward_score_csv_path: /data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/FlowGRPO-PickScore/ckpt-0/evaluation_results.jsonl
method_reward_score_csv_path: /data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/FlowGRPO-PickScore-next-random_9984_images_no_anime_pickscore_002-hpdv3_all-inpainting/ckpt-300/evaluation_results.jsonl

Win Rates Summary:
Evaluation Model  Win Rate  Wins  Ties  Losses  Total
             avg     0.988   494     0       6    500
           hpsv3     0.748   374     0     126    500
       aesthetic     0.604   302     0     198    500
        ma-agiqa     0.602   301     0     199    500
     imagereward     0.548   274     0     226    500
      diffdoctor     0.527    89   349      62    500
       pickscore     0.518   259     0     241    500
            deqa     0.515   248    19     233    500
  aesthetic_v2_5     0.428   214     0     286    500
